- RetrieverDoc - https://docs.langchain.com/oss/python/integrations/retrievers

# 📚 Guide to Information Retrievers in RAG

Retrievers are the backbone of **Retrieval-Augmented Generation (RAG)**. They bridge the gap between an LLM's static training and dynamic, real-world data.

---

### 🔹 1. Wikipedia Retriever
> **Best for:** General knowledge and factual queries.
- **Mechanism:** Queries the live Wikipedia API.
- **Highlight:** It doesn't require pre-indexing your own data; it uses the world's largest open encyclopedia as its vector-less database.

### 🔹 2. Vector Store Retriever
> **Best for:** Core semantic search within custom datasets.
- **Mechanism:** Uses **Embeddings** (numerical vectors) to find documents based on 'meaning' rather than keywords.
- **Key Components:** Requires an Embedding Model (e.g., OpenAI) and a Vector Database (e.g., Chroma, FAISS).

### 🔹 3. MMR (Maximal Marginal Relevance)
> **Best for:** Reducing redundancy and avoiding repetitive answers.
- **The Difference:** Standard search finds the *most similar* items. MMR finds items that are **similar to the query but different from each other**.
- **Key Setting:** `lambda_mult`. Set it closer to `0` for maximum diversity, or `1` for pure similarity.

### 🔹 4. MultiQuery Retriever
> **Best for:** Overcoming poorly phrased or narrow user prompts.
- **The Difference:** It uses an LLM to rewrite your single question into **3-5 variations**. It then retrieves documents for *all* of them.
- **Why it matters:** It catches relevant documents that might not have shared exact keywords with your original query.

### 🔹 5. Contextual Compression Retriever
> **Best for:** Reducing noise, saving tokens, and increasing accuracy.
- **The Difference:** Normal retrievers return whole paragraphs. This retriever uses an LLM to **extract only the relevant sentences** from those paragraphs before showing them to the final model.
- **Result:** You pass much cleaner, 'compressed' data to your LLM.

---

### ⚖️ Key Comparison Table

| Retriever Type | Input Handling | Output Quality | Primary Strength |
| :--- | :--- | :--- | :--- |
| **Standard Vector** | Single query | Raw document chunks | Speed and simplicity |
| **MMR** | Single query | Diverse document chunks | Eliminates repetitive info |
| **MultiQuery** | Multiple LLM-generated queries | Broad document set | High recall (misses nothing) |
| **Compression** | Post-processing | Summarized/Extracted snippets | High precision (removes noise) |

### 🚀 Modern Techniques to Know
- **Hybrid Search:** Combining Keyword (BM25) + Vector search.
- **Re-ranking:** Fetching 20 documents quickly, then using a 'Cross-Encoder' to pick the best 5.
- **Self-RAG:** The model decides for itself if it actually needs to retrieve information or if it can answer on its own.

**1. Wikipedia Retriever**

In [1]:
from langchain_community.retrievers import WikipediaRetriever

In [3]:
# Initialize the retriever (optional: set language and top_k(top k results))
retriever = WikipediaRetriever(top_k_results=2, lang="en")


# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)


In [7]:
docs

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as th

In [ ]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display



--- Result 1 ---
Content:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,

**2. Vector Store Retriever**

In [9]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

from dotenv import load_dotenv

load_dotenv()

True

In [10]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [11]:
# Step 2: Initialize embedding model
embedding_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001', dimensions=32)

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [12]:
# Step 4: Convert vectorstore into a retriever with serching top 2 result
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [13]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [14]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [ ]:
# without retriever, using directly vector store by similarity search to get result
results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


- But the diference is between vector store similarity search and by using it as retriever is huge.

1. When using retiriever we can use various strategies to search and retrieve the information whereas when using normal vector store only we can use semantic search strategy(which is default for vector store retriever also)

2. As retriever is runnable so we have flexiblity to integrate this with chains but we can't do same with vector store.

**3. MMR - Maximum Marginal Retrivers** 

- It retrives the result that are not only relavent to user query but also different from each other(means removes redudant results)

In [22]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
     Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [23]:
# Step 2: Create the FAISS vector store from documents
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [24]:
# Enable MMR in the retriever

#Here is the example that if we use vector store as retriver than can leverage various strategy, like here used mmr
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [21]:
query = "What is langchain?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 3 ---
MMR helps you get diverse results when doing similarity search.


**4. Multiquery Retriever**

- It used when a query is ambiguous/confusing, when single query might not capture all the results.
- it is usefull when we want all the relavent result by avoiding the ambiguity from the query.

In [30]:
# Change from: from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_google_genai import ChatGoogleGenerativeAI

In [28]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [ ]:
vectorstore = Chroma.from_documents(documents=all_docs, embedding=embedding_model)

# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_tokens=256)
)

In [36]:
# Query
query = "How to improve energy levels and maintain balance?"

In [37]:
# Retrieve results

#by simple similarity search
similarity_results = similarity_retriever.invoke(query)

#by multiquery, to remove ambiguity and retiriev more relavent result
multiquery_results= multiquery_retriever.invoke(query)

In [38]:
print('Normal Similarity Search Results\n')
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

print('MultiQuery Results\n')
for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

Normal Similarity Search Results


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 3 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 4 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 5 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
******************************************************************************************************************************************************
MultiQuery Results


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.


**5. Contextual Compression Retriver**

- It used when we are having a document where in a single document itself, we can have more than 1 topic information.
- In this kind of situation this retriver helps to compress the single retrived result itself to give only relavent information and removes the irrelavent info which is not related to the topic asked in user query.


- Example - 
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""

        *This single doc talking about photosynthesis as well ass tourist*

In [40]:
# Change these lines:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [41]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [42]:
# We can use FAISS vector also
vectorstore = Chroma.from_documents(docs, embedding_model)

#create base retriver , i.e. vector store retriver
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Set up the compressor using an LLM
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_tokens=256)
compressor = LLMChainExtractor.from_llm(llm)

# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [ ]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [44]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 2 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 3 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 4 ---
Photosynthesis does not occur in animal cells.


In [47]:
#we can integrate mmr to keep unique output

#create base retriver , i.e. vector store retriver
base_retriever = vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "lambda_mult": 0.5})

# Set up the compressor using an LLM
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_tokens=256)
compressor = LLMChainExtractor.from_llm(llm)

# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [48]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 2 ---
Photosynthesis is the process by which green plants convert sunlight into energy.
